# "Multi-Label Classification"
> Using Fastai to for Multi-Label classification.

- toc: true
- badges: true
- comments: true
- use_math: true    
- categories: [fastai, jupyter]

# Introduction

In a previous [post](https://nasheqlbrm.github.io/blog/fastai/gradio/jupyter/2021/07/31/Bear-Classification-And-Gradio.html) we saw our Bear classifier confidently predict that an image of a Maine Coone was a Teddy bear. The problem is that if any of the activations for some class, in a multi-class classification framework, is higher than the rest then the exponential (in the softmax) ensures that class receives a probability score close to one. This is an issue when the object being classified is not among the ones that the classifier was originally trained upon. 

In this post we use multi-label classification to solve this problem. A multi-label classification problem (as opposed to multi-class classification) is one where there can be multiple labels, instead of there being a single correct class, for every object.

We want our multi-label classifier to predict the different labels present in the object or no label if it is not confident about the absence of all of them. For this we switch to using Binary Cross Entropy as our loss function and will have to come up with a threshold probability for our classifier to use for inference.

## Binary Cross Entropy

In [1]:
from fastai.vision.all import *

Suppose the following are the activations, on a single training example, for the multiple classes of a multi-label classification problem. Pretend that we can have as many as three labels for each image.

In [2]:
activations = torch.randn(1,3)*3
activations

tensor([[ 1.3737, -2.8129, -3.1691]])

Next suppose the correct labels in this case are the first and third label. A one-hot encoded representation of the labels for this single training example is as follows.

In [3]:
targets = tensor([[1,0,1]])
targets

tensor([[1, 0, 1]])

The first step is to take the sigmoid activations for each label to convert each activation into a probability score.

In [4]:
sigmoid_activations = activations.sigmoid()
sigmoid_activations

tensor([[0.7980, 0.0566, 0.0403]])

The next step is to compute the binary cross entropy loss using

In [5]:
bce_loss = -torch.where(targets==1, sigmoid_activations, 1-sigmoid_activations).log()
bce_loss

tensor([[0.2257, 0.0583, 3.2102]])

When a label is present, in the target, we take the sigmoid activation corresponding to that label. Otherwise, when the label is absent, we take one minus the that sigmoid activation. For the latter case we are effectively taking the _sum_ of sigmoid activations predicted for the _remaining_ labels. By doing this we are asking for the confidence that the classifier places on the label being absent. 

Thus the binary cross entropy function summarizes the "correctness" of the classifer across the presence and absence of each possible label in a training example. Contrast this with the use of [cross entropy loss](https://nasheqlbrm.github.io/blog/fastai/pytorch/jupyter/2021/08/07/Cross-Entropy-Loss-PyTorch.html) in multi-class classification where we only care about the probability being predicted for the single correct class the training example.

In [6]:
results = torch.reshape(torch.cat([activations, sigmoid_activations, targets, bce_loss], 0),(4,3))
pd.DataFrame(results, index=['activations', 'sigmoid','target','loss'])

,0,1,2
activations,1.373679,-2.812911,-3.169061
sigmoid,0.797974,0.056630,0.040347
target,1.000000,0.000000,1.000000
loss,0.225679,0.058297,3.210244


# Bear Images

Let's assemble images of bears.

In [8]:
#collapse-hide
from jmd_imagescraper.core import *

def scrape_images(path, labels, search_suffix, erase_dir=True, max_images=20):
    if erase_dir:
        !rm -rf {path}
    
    if not path.exists():
        path.mkdir(parents=True)
    
    for some_label in labels:
        duckduckgo_search(path, some_label,\
                          f'{some_label} {search_suffix}', max_results=max_images)
        
    filenames = get_image_files(path)
    failed = verify_images(filenames)
    
    failed.map(Path.unlink);
    if failed != []:
        _ = [filenames.remove(f) for f in failed]
    
    # To avoid Transparency warnings, convert PNG images to RGBA
    # https://forums.fast.ai/t/errors-when-training-the-bear-image-classification-model/83422/9
    converted = L()
    for image in filenames:
        if '.png' in str(image):
            im = Image.open(image)
            converted.append(image)  # old file name before resaving
            im.convert("RGBA").save(f"{image}2.png")    
            
    converted.map(Path.unlink); # delete originals
    
    total_images = len(get_image_files(path))
    print(f"After checking for issues, {total_images} (total) images remain.")
    
    return path

In [9]:
labels = 'grizzly','black','teddy'
path = scrape_images(Path('/data/kaushik/bears'), labels, 'bear', max_images=100)

Duckduckgo search: grizzly bear


Duckduckgo search: black bear


Duckduckgo search: teddy bear


After checking for issues, 300 (total) images remain.
